# FinBERT Sentiment Model

This notebook begins Phase 2 by preparing a finance-pretrained transformer for sentiment classification. It uses the same cleaned Financial PhraseBank split as the Phase 1 baseline.

In [ ]:
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

In [ ]:
MODEL_NAME = "yiyanghkust/finbert-pretrain"
RANDOM_STATE = 42
MAX_LENGTH = 128
LABEL_NAMES = {0: "negative", 1: "neutral", 2: "positive"}

device = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
device

## Reproduce the Phase 1 data split

In [ ]:
dataset = load_dataset(
    "takala/financial_phrasebank",
    "sentences_75agree",
    trust_remote_code=True,
)

df = (
    dataset["train"]
    .to_pandas()
    .drop_duplicates(subset="sentence")
    .reset_index(drop=True)
)

train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=df["label"],
)

split_summary = pd.DataFrame(
    {
        "rows": [len(train_df), len(test_df)],
        "negative_share": [
            (train_df["label"] == 0).mean(),
            (test_df["label"] == 0).mean(),
        ],
        "neutral_share": [
            (train_df["label"] == 1).mean(),
            (test_df["label"] == 1).mean(),
        ],
        "positive_share": [
            (train_df["label"] == 2).mean(),
            (test_df["label"] == 2).mean(),
        ],
    },
    index=["train", "test"],
).round(3)

split_summary

## Transformer tokenization

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

example_sentence = train_df.iloc[0]["sentence"]
example_tokens = tokenizer.tokenize(example_sentence)
example_ids = tokenizer.convert_tokens_to_ids(example_tokens)

pd.DataFrame(
    {"token": example_tokens, "token_id": example_ids}
).head(30)

In [ ]:
encoded_example = tokenizer(
    example_sentence,
    max_length=MAX_LENGTH,
    truncation=True,
    padding="max_length",
    return_tensors="pt",
)

{name: tuple(values.shape) for name, values in encoded_example.items()}

Unlike the Phase 1 unigram tokenizer, the FinBERT tokenizer can split unfamiliar words into subword pieces. It adds model-specific special tokens and returns token IDs plus an attention mask.

In [ ]:
token_lengths = train_df["sentence"].map(
    lambda sentence: len(tokenizer.tokenize(sentence)) + 2
)

pd.Series(
    {
        "median_tokens": token_lengths.median(),
        "95th_percentile": np.percentile(token_lengths, 95),
        "maximum_tokens": token_lengths.max(),
        "truncated_at_128": (token_lengths > MAX_LENGTH).sum(),
    }
).astype(int)

The token-length check verifies whether a maximum length of 128 preserves nearly all sentences before fine-tuning. The next stage will create encoded training and validation datasets and attach a three-class classification head.